In [25]:
import os
import sys
sys.path.append(os.path.join('..', '..'))

from scripts.data_loaders.TIMIT import TIMITDataset
from scripts.data_loaders.EpaDB import EpaDBDataset
from scripts.data_loaders.PSST import PSSTDataset
from scripts.data_loaders.SpeechOcean import SpeechOceanDataset
from scripts.data_loaders.ISLE import ISLEDataset
from scripts.core.text import english2ipa_espeak

from scripts.eval.evaluate import evaluate

from collections import namedtuple
import csv
import pandas as pd
from IPython.display import clear_output
clear_output()

In [ ]:
TestSet = namedtuple('TestSet', ['name', 'data'])

testsets = [
    TestSet('TIMIT', TIMITDataset(split='test', include_text=True)),
    TestSet('EpaDB', EpaDBDataset(split='test', include_text=True)),
    TestSet('PSST', PSSTDataset(split='test', force_offline=True, include_text=True)),
    TestSet('SpeechOcean', SpeechOceanDataset(split='test', include_text=True)),
    TestSet('ISLE', ISLEDataset(split="all", include_text=True))
]

In [ ]:
csv_path = "../../.data/ipa_g2p_espeak_scores.csv"

with open(csv_path, 'a', newline='') as f:
    writer = csv.writer(f)
    
    if os.path.getsize(csv_path) == 0:
        writer.writerow(["dataset", "groundtruth", "text", "g2p_espeak", "per", "fer"])

    for datasetname, dataset in testsets:
        for sample in dataset:
            ipa, audio, text = sample
            g2p_espeak = english2ipa_espeak(text)
            per, fer = evaluate(ipa, g2p_espeak)
            writer.writerow([datasetname, ipa, text, g2p_espeak, f"{per:.2f}", f"{fer:.2f}"])

In [30]:
df = pd.read_csv(csv_path)
print(df.groupby("dataset")[["per", "fer"]].mean())
print(df.groupby("dataset")[["per", "fer"]].mean().mean())


                  per       fer
dataset                        
EpaDB        0.339905  0.039052
ISLE         0.396421  0.035227
PSST         0.691334  0.258083
SpeechOcean  0.162590  0.023005
TIMIT        0.269113  0.049679
per    0.371873
fer    0.081009
dtype: float64
